In [2]:
import mlflow
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.model_selection import train_test_split
import joblib
from sklearn.metrics import r2_score as R2

In [3]:
df = pd.read_csv('D:/Maga/6th module/MLOps data/final_df.csv', index_col=0)

df

,brand,categories,colors,price,is_available,condition,is_sale,source,shipping,is_special_offer,is_returnable,manufacturer
1284,Forever Young,shoes,black,29.99,1,New,1,Overstock.com,No,1,0,RB-1904
1284,Forever Young,clothing,black,29.99,1,New,1,Overstock.com,No,1,0,RB-1904
1286,Forever Young,shoes,black,49.88,1,New,1,Overstock.com,No,1,0,RB-1904
1286,Forever Young,clothing,black,49.88,1,New,1,Overstock.com,No,1,0,RB-1904
7862,Hush Puppies,shoes,black,23.54,1,New,1,Overstock.com,No,1,0,H509362-635-75 M US
...,...,...,...,...,...,...,...,...,...,...,...,...
33797,Novica,clothing,red,39.99,1,New,1,Overstock.com,No,0,0,276629
33797,Novica,shoes,red,39.99,1,New,1,Overstock.com,No,0,0,276629
33798,Novica,accessories,red,35.99,1,New,0,Overstock.com,No,0,0,276629
33798,Novica,clothing,red,35.99,1,New,0,Overstock.com,No,0,0,276629


In [4]:
# Категориальные признаки:
# brand
# condition
# source
# shipping
# manufacturer
# category
# color

categorical_cols = df.dtypes[df.dtypes == 'object'].index
categorical_cols

Index(['brand', 'categories', 'colors', 'condition', 'source', 'shipping',
       'manufacturer'],
      dtype='object')

In [5]:
for col in categorical_cols:
    le = LabelEncoder()
    save_path = f'D:/Maga/6th module/MLOps/ml-model/ml/data/encoders/{col}_classes.npy'
    le.classes_ = np.load(save_path, allow_pickle=True)
    df[col] = le.transform(df[col])
    
df

,brand,categories,colors,price,is_available,condition,is_sale,source,shipping,is_special_offer,is_returnable,manufacturer
1284,281,2,1,29.99,1,1,1,12,1,1,0,1831
1284,281,1,1,29.99,1,1,1,12,1,1,0,1831
1286,281,2,1,49.88,1,1,1,12,1,1,0,1831
1286,281,1,1,49.88,1,1,1,12,1,1,0,1831
7862,324,2,1,23.54,1,1,1,12,1,1,0,1355
...,...,...,...,...,...,...,...,...,...,...,...,...
33797,500,1,6,39.99,1,1,1,12,1,0,0,293
33797,500,2,6,39.99,1,1,1,12,1,0,0,293
33798,500,0,6,35.99,1,1,0,12,1,0,0,293
33798,500,1,6,35.99,1,1,0,12,1,0,0,293


In [6]:
# выделение столбцов-признаков и целевой переменной
X, y = df.drop(columns=['price']).to_numpy(), df['price'].to_numpy()

In [7]:
# разделение на обучающее и тестовое множества
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [8]:
scaler_filename = 'D:/Maga/6th module/MLOps/ml-model/ml/data/scalers/y_scaler.save'
scaler = joblib.load(scaler_filename)

y_train = scaler.transform(y_train.reshape(-1, 1))

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

models = []

models.append(("LinearRegression:", LinearRegression()))
models.append(("DecisionTreeRegressor:", DecisionTreeRegressor()))
models.append(("RandomForestRegressor:", RandomForestRegressor()))
models.append(("MLPRegressor:", MLPRegressor()))
models.append(("XGBRegressor:", XGBRegressor()))

models

[('LinearRegression:', LinearRegression()),
 ('DecisionTreeRegressor:', DecisionTreeRegressor()),
 ('RandomForestRegressor:', RandomForestRegressor()),
 ('MLPRegressor:', MLPRegressor()),
 ('XGBRegressor:',
  XGBRegressor(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, gamma=None,
               gpu_id=None, grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=None, max_bin=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, n_estimators=100, n_jobs=None,
               num_parallel_tree=None, predictor=None, random_state=None,
               reg_alpha=None, reg_lambda=None, ...))]

In [ ]:
# установка места сохранения эксперементов
mlflow.set_tracking_uri("file:///D:/Maga/6th module/MLOps/mlflow/mlruns")

# создаем эксперимент
expname = 'Laba-exp-base-models-again'
exp_id = mlflow.create_experiment(expname)
exp_id

for name, model in models:
    model.fit(X_train, y_train)
    predict_train = model.predict(X_train)
    predict_test_norm = model.predict(X_test)
    predict_test = scaler.inverse_transform(predict_test_norm.reshape(-1, 1))
    score = R2(y_test, predict_test)
    with mlflow.start_run(experiment_id=exp_id, run_name=name):
        mlflow.log_param('model_name', name)
        mlflow.log_metric("R2_score", score)

C:\Users\alexsp\AppData\Local\Temp\ipykernel_15908\1572462777.py:10: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X_train, y_train)
D:\Apps\lib\site-packages\sklearn\utils\validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)


Создание модели для развертывания

In [12]:
# установка места сохранения эксперементов
mlflow.set_tracking_uri("file:///D:/Maga/6th module/MLOps/mlflow/mlruns")

# создаем эксперимент
expname = 'RandomForestRegressor'
exp_id = mlflow.create_experiment(expname)

model = RandomForestRegressor()
with mlflow.start_run(experiment_id=exp_id, run_name=expname):
    # автоматом сохраняет model's parameters, metrics, artifacts
    mlflow.sklearn.autolog()
    model.fit(X_train, y_train)
    run_id = mlflow.active_run().info.run_id

print(exp_id)
print(run_id)

2025/01/13 22:50:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "D:\Apps\lib\site-packages\mlflow\utils\autologging_utils\safety.py:570: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel()."
2025/01/13 22:50:30 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\alexsp\AppData\Local\Temp\tmpxbzvoeof\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==0.24.2', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 


815697592354651329
d46c1d66ec27463ba6d01ab5adce5ed0


In [13]:
exp_id = '815697592354651329'
run_id = 'd46c1d66ec27463ba6d01ab5adce5ed0'

model_name = 'app_model'

# стандартный путь к каталогу с моделями
artifact_path = 'model'
model_uri = f'runs:/{run_id}/{artifact_path}'

model_details_tf = mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'app_model'.
Created version '1' of model 'app_model'.


Убедимся в том же ноутбуке, что это будет работать, прежде чем интегрировать модель в приложение.

In [14]:
logged_model = 'runs:/d46c1d66ec27463ba6d01ab5adce5ed0/model'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: model
  flavor: mlflow.sklearn
  run_id: d46c1d66ec27463ba6d01ab5adce5ed0

In [18]:
# Predict on a Pandas DataFrame.
import pandas as pd

data = [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
predicted_value_norm = loaded_model.predict(pd.DataFrame(data))
predicted_price = scaler.inverse_transform(predicted_value_norm.reshape(-1, 1))
predicted_price

array([[141.9301875]])